# Phase I — Data Cleaning
**Project:** Ice Hockey Talent Scouting (KDDM)

This notebook performs Step 2 of the Phase I roadmap and the cleaning that follows from it:

1. Load each of the 7 raw files with the correct reader.
2. Inspect each (`shape`, `dtypes`, `head`, `describe`, `isnull`).
3. Document findings inline.
4. Apply per-file cleaning.
5. Combine identity records and persist clean CSVs to `data/clean/`.

## 0. Setup

In [19]:
from pathlib import Path
import re

import numpy as np
import pandas as pd

ROOT = Path.cwd()
DATA = ROOT / "data"
OUT = DATA / "clean"
OUT.mkdir(parents=True, exist_ok=True)

pd.set_option("display.max_columns", None)
pd.set_option("display.width", 200)
pd.set_option("display.max_colwidth", 60)


## 1. Load each file with the correct reader

| File | Reader | Notes |
|---|---|---|
| `contracts_competition.pkl` | `read_pickle` | binary pickle |
| `hockey_scouting_notes.csv` | `read_csv` | comma-separated |
| `identity_card_0.tsv` | `read_csv(sep="\t")` | Latin headers, Roman-numeral IDs |
| `identity_card_1.csv` | `read_csv` | English headers |
| `medical_information.xlsx` | `read_excel(engine="openpyxl")` | header is split across two rows |
| `moms_notes.json` | `read_json` | array of objects |
| `performance.tsv` | `read_csv(sep="\t")` | tab-separated |

In [20]:
contracts = pd.read_pickle(DATA / "contracts_competition.pkl")
scouting  = pd.read_csv(DATA / "hockey_scouting_notes.csv")
id0       = pd.read_csv(DATA / "identity_card_0.tsv", sep="\t")
id1       = pd.read_csv(DATA / "identity_card_1.csv")
medical   = pd.read_excel(DATA / "medical_information.xlsx", engine="openpyxl")
moms      = pd.read_json(DATA / "moms_notes.json")
perf      = pd.read_csv(DATA / "performance.tsv", sep="\t")

frames = {
    "contracts_competition": contracts,
    "hockey_scouting_notes": scouting,
    "identity_card_0":       id0,
    "identity_card_1":       id1,
    "medical_information":   medical,
    "moms_notes":            moms,
    "performance":           perf,
}
{name: df.shape for name, df in frames.items()}

{'contracts_competition': (10000, 8),
 'hockey_scouting_notes': (10000, 7),
 'identity_card_0': (888, 8),
 'identity_card_1': (9112, 8),
 'medical_information': (10002, 11),
 'moms_notes': (10000, 13),
 'performance': (10000, 29)}

## 2. Inspect each file

A single helper, then one cell per file.

In [21]:
def inspect(name, df):
    print(f"=== {name} | shape: {df.shape} ===")
    print("\n[dtypes]")
    print(df.dtypes)
    print("\n[head]")
    print(df.head())
    num = df.select_dtypes("number")
    if not num.empty:
        print("\n[describe — numeric]")
        print(num.describe())
    obj = df.select_dtypes("object")
    if not obj.empty:
        print("\n[describe — object]")
        print(obj.describe())
    print("\n[isnull %]")
    print((df.isnull().sum() / len(df) * 100).round(2))
    print(f"\n[duplicate rows]: {df.duplicated().sum()}")

In [22]:
inspect("contracts_competition", contracts)

=== contracts_competition | shape: (10000, 8) ===

[dtypes]
international_id              int64
contracts_signed            float64
salary                      float64
captain                      object
won_championship             object
jersey_number               float64
draft_year                  float64
number_of_previous_teams    float64
dtype: object

[head]
   international_id  contracts_signed      salary captain won_championship  jersey_number  draft_year  number_of_previous_teams
0              6032               5.0   669578.77   False             True           48.0      2022.0                      12.0
1              3654               5.0  1647102.80   False            False           10.0      2013.0                      15.0
2              8729               6.0  1678593.48   False            False           63.0      2016.0                      14.0
3              2300               1.0   927858.79   False            False           46.0      2015.0                 

In [23]:
inspect("hockey_scouting_notes", scouting)

=== hockey_scouting_notes | shape: (10000, 7) ===

[dtypes]
international_id      int64
position             object
dominant_hand        object
experience_level     object
years_played        float64
years_pro           float64
scout_notes          object
dtype: object

[head]
   international_id    position dominant_hand experience_level  years_played  years_pro                                                  scout_notes
0              2991     defense          left          veteran           8.0        8.0  effort varies throughout the game, awkward stride mechanics
1              3042  right wing         right          veteran           0.0        0.0  a consistent performer who can be counted on in any situ...
2              3761  right wing         right          veteran           0.0        0.0                                    late support on breakouts
3              6229   left wing          left           rookie           1.0        1.0                                       

In [24]:
inspect("identity_card_0", id0)

=== identity_card_0 | shape: (888, 8) ===

[dtypes]
numerus_internationalis_ad_identitatem     object
numerus_identificationis_ad_medicinam      object
praenomen                                  object
cognomen                                   object
sexus                                      object
aetas_annorum                             float64
urbs_natalis                               object
natio                                      object
dtype: object

[head]
  numerus_internationalis_ad_identitatem numerus_identificationis_ad_medicinam praenomen     cognomen   sexus  aetas_annorum urbs_natalis    natio
0                                 CCLXXV  039e0ab7-e36d-4bc7-a0b1-3e93e8ec73e9   Keithus     Alleeius    male           23.0       vienna  austria
1                                  MMDLX  070dfb40-76c9-4894-90e5-7463d060d49b    Alyssa      Kohlius       f           31.0       Vienna  austria
2                             MMCCCXXIII  fa06ed07-6f09-4ec2-9f11-a70917307270     Em

In [25]:
inspect("identity_card_1", id1)

=== identity_card_1 | shape: (9112, 8) ===

[dtypes]
international_id      int64
medical_id           object
first_name           object
last_name            object
gender               object
age                 float64
birth_city           object
nationality          object
dtype: object

[head]
   international_id                            medical_id first_name last_name  gender   age birth_city nationality
0              3731  c408fbd0-cac8-4950-92e8-ea7c73fa1361    Preston      Mick    male  23.0  Trondheim      norway
1              1628  66522ba9-e5a0-4f3c-821f-41ed0d65d3ca     Wilbur     Moore    male  20.0  Marseille       U.S.A
2              7448  e417f30c-187d-4ef9-8c41-ee3bf255baea    Jeffrey  Kindrick    male  25.0     Boston         usa
3              3922  feb6dc99-908d-44c7-be73-397404be5bf1    Vanessa   Hawkins   other  31.0    Liberec     czechia
4              9394  5336e053-fdd3-4312-b475-d11c6bb783d8    Crystal     Green  female  22.0     Moscow      russia

[des

In [26]:
inspect("medical_information", medical)

=== medical_information | shape: (10002, 11) ===

[dtypes]
Unnamed: 0                                                              object
Date: 2026-03-31 | Meta Data: "I'm a doctor, not an engineer" ~McCoy    object
Unnamed: 2                                                              object
Unnamed: 3                                                              object
Unnamed: 4                                                              object
Unnamed: 5                                                              object
Unnamed: 6                                                              object
Unnamed: 7                                                              object
Unnamed: 8                                                              object
Unnamed: 9                                                              object
Unnamed: 10                                                             object
dtype: object

[head]
                             Unnamed: 0 Date: 2026

In [27]:
inspect("moms_notes", moms)

=== moms_notes | shape: (10000, 13) ===

[dtypes]
first_name               object
last_name                object
age                     float64
birth_city               object
school_grade            float64
years_in_usa            float64
eye_color                object
stuffed_animal_name      object
favourite_board_game     object
personal_notes           object
favourite_child_info     object
favourite_tv_show        object
favourite_food           object
dtype: object

[head]
    first_name last_name   age birth_city  school_grade  years_in_usa eye_color stuffed_animal_name favourite_board_game                  personal_notes  \
0       Edward    Battey  21.0  Stockholm          3.51           0.0     HAZEL                Dawn              Attack!                            None   
1     Winifred      Bien  19.0       Oslo          3.02           2.0     HAZEL                John   Dungeons & Dragons       gently reflective in life   
2         None      None   NaN       None   

In [28]:
inspect("performance", perf)

=== performance | shape: (10000, 29) ===

[dtypes]
international_id                int64
goals                         float64
assists                       float64
num_of_shots                  float64
shot_speed                    float64
shot_attempts                 float64
shooting_percentage           float64
high_danger_shots             float64
medium_danger_shots           float64
low_danger_shots              float64
save_percentage               float64
winning_goals                 float64
power_play_time               float64
power_play_goals              float64
time_on_ice                   float64
faceoff_win_percentage        float64
puck_touches                  float64
puck_recoveries               float64
puck_possession_time          float64
penalty_kill_time             float64
penality_minutes              float64
penalties_taken               float64
time_between_penalties        float64
goals_against_total           float64
goals_against_average         float64

### Findings (feeds Phase I slides 2–3 and 6–7)

**`contracts_competition.pkl`** — 10 000 × 8, keyed by `international_id`. `captain` and `won_championship` are stored as Python booleans inside object dtype. Low null rates (<2 %).

**`hockey_scouting_notes.csv`** — 10 000 × 7, keyed by `international_id`. `dominant_hand` is ~6.5 % null and `scout_notes` is ~10 % null. Position/hand/level need lower-casing.

**`identity_card_0.tsv`** — only **888 rows**. Column names are Latin (`praenomen`, `cognomen`, `sexus`, …). `international_id` is encoded as **Roman numerals**. Last names carry the Latinized `-ius` suffix. `gender` mixes `m` / `male` / `f` / `female`. `birth_city` mixes case (`vienna`, `Vienna`, `OSTRAVA`). `age` has impossible outliers (max **390**).

**`identity_card_1.csv`** — 9 112 rows in English. Same gender / city / nationality inconsistencies (`U.S.A` vs `usa`). `age` max is **455** — same outlier pattern. Together id_0 + id_1 ≈ 10 000 unique players.

**`medical_information.xlsx`** — header is **split across two rows**: `medical_id` lives in row 1 col 0 while the other column names live in row 0 cols 1–10. After re-reading: `height` is mixed `"1.79m"` / `"180.1135cm"`, `weight` includes the `kg` suffix and the literal token `"nankg"`, `body_fat_percentage` is sometimes `"20.87%"`. The `medical_information` column is ~85 % null.

**`moms_notes.json`** — 10 000 × 13 with **36 duplicate rows** and **3 entirely constant columns** (`favourite_child_info`, `favourite_tv_show`, `favourite_food`). `personal_notes` is 25 % null. `age` carries the same impossible-value outliers as the identity files. **No id column** — must be joined on `(first_name, last_name, age, birth_city)`.

**`performance.tsv`** — 10 000 × 29, the cleanest file. All numeric, very low nulls, keyed by `international_id`. Note typo `penality_minutes` (sic).

## 3. Cleaning

Each subsection produces one `*_clean` DataFrame. Final cell concatenates identities and writes everything to `data/clean/`.

### 3.1 `moms_notes` — drop constant columns and duplicates

In [29]:
constant_cols = [c for c in moms.columns if moms[c].nunique(dropna=False) <= 1]
print("constant columns dropped:", constant_cols)

moms_clean = (
    moms
    .drop(columns=constant_cols)
    .drop_duplicates()
    .reset_index(drop=True)
)
moms_clean.shape

constant columns dropped: ['favourite_child_info', 'favourite_tv_show', 'favourite_food']


(9964, 10)

### 3.2 `contracts_competition` — coerce booleans, sanity-check `draft_year`

In [30]:
contracts_clean = contracts.copy()

BOOL_MAP = {True: True, False: False, "True": True, "False": False, "true": True, "false": False}
for col in ["captain", "won_championship"]:
    contracts_clean[col] = contracts_clean[col].map(BOOL_MAP).astype("boolean")

# draft_year minimum was 1850 — implausible. NHL drafts began in 1963.
implausible = contracts_clean["draft_year"] < 1960
print(f"implausible draft_year set to NaN: {implausible.sum()}")
contracts_clean.loc[implausible, "draft_year"] = np.nan

contracts_clean.dtypes

implausible draft_year set to NaN: 40


international_id              int64
contracts_signed            float64
salary                      float64
captain                     boolean
won_championship            boolean
jersey_number               float64
draft_year                  float64
number_of_previous_teams    float64
dtype: object

### 3.3 `identity_card_0` — Latin → English headers, Roman → integer IDs

Decoder is the standard subtractive-rule Roman numeral parser (e.g. `MMDLX` = 2560).

In [31]:
LATIN_TO_EN = {
    "numerus_internationalis_ad_identitatem": "international_id",
    "numerus_identificationis_ad_medicinam":  "medical_id",
    "praenomen":      "first_name",
    "cognomen":       "last_name",
    "sexus":          "gender",
    "aetas_annorum":  "age",
    "urbs_natalis":   "birth_city",
    "natio":          "nationality",
}

ROMAN = {"M": 1000, "D": 500, "C": 100, "L": 50, "X": 10, "V": 5, "I": 1}

def roman_to_int(s):
    if not isinstance(s, str):
        return pd.NA
    total, prev = 0, 0
    for ch in reversed(s.upper().strip()):
        v = ROMAN.get(ch)
        if v is None:
            return pd.NA
        total += -v if v < prev else v
        prev = v
    return total

id0_clean = id0.rename(columns=LATIN_TO_EN).copy()
id0_clean["international_id"] = id0_clean["international_id"].map(roman_to_int).astype("Int64")
# strip the Latin -ius suffix that was bolted onto last names
id0_clean["last_name"] = id0_clean["last_name"].str.replace(r"ius$", "", regex=True)
id0_clean[["international_id", "last_name"]].head()

,international_id,last_name
0,275,Allee
1,2560,Kohl
2,2323,Belanger
3,7708,Fiedler
4,4062,Mulligan


### 3.4 Shared normalization for both identity files (gender, city, nationality, age outliers)

In [32]:
GENDER_MAP = {
    "m": "male", "male": "male",
    "f": "female", "female": "female",
    "other": "other",
}

NATIONALITY_MAP = {
    "u.s.a": "usa", "u.s.a.": "usa", "usa": "usa",
    "united states": "usa", "united states of america": "usa",
}

def norm_gender(g):
    if not isinstance(g, str):
        return np.nan
    return GENDER_MAP.get(g.strip().lower(), g.strip().lower())

def norm_nationality(s):
    if not isinstance(s, str):
        return np.nan
    key = s.strip().lower().rstrip(".")
    return NATIONALITY_MAP.get(key, key)

id1_clean = id1.copy()

for df in (id0_clean, id1_clean):
    df["gender"]      = df["gender"].map(norm_gender)
    df["birth_city"]  = df["birth_city"].astype("string").str.title()
    df["nationality"] = df["nationality"].map(norm_nationality)
    df.loc[df["age"] > 60, "age"] = np.nan  # drop the 390 / 455 garbage

moms_clean.loc[moms_clean["age"] > 60, "age"] = np.nan

print("id0 genders:", id0_clean["gender"].value_counts(dropna=False).to_dict())
print("id1 genders:", id1_clean["gender"].value_counts(dropna=False).to_dict())
print("id0 age max:", id0_clean["age"].max(), "| id1 age max:", id1_clean["age"].max())

id0 genders: {'male': 403, 'female': 403, 'other': 82}
id1 genders: {'male': 4154, 'female': 4089, 'other': 832, nan: 37}
id0 age max: 38.0 | id1 age max: 39.0


### 3.5 Combine the two identity files

In [33]:
identity = (
    pd.concat([id0_clean, id1_clean], ignore_index=True)
      .drop_duplicates(subset=["international_id"])
      .reset_index(drop=True)
)
identity["international_id"] = identity["international_id"].astype("Int64")
print("combined identity:", identity.shape)
print("unique medical_ids:", identity["medical_id"].nunique())

combined identity: (10000, 8)
unique medical_ids: 10000


### 3.6 `medical_information` — re-read with proper headers, parse units

The file has a junk metadata row at the top, real column names on row 1, and `medical_id` parked alone in row 2 col 0. Real data starts at row 3.

In [34]:
med_raw = pd.read_excel(DATA / "medical_information.xlsx", engine="openpyxl", header=None)
cols = ["medical_id"] + med_raw.iloc[1, 1:].tolist()
medical_clean = med_raw.iloc[3:].reset_index(drop=True).copy()
medical_clean.columns = cols
medical_clean.head()

,medical_id,height,weight,age_in_years,shoe_size,body_fat_percentage,fitness_level,sprint_time,medical_information,return_date,physician_signature
0,368bcbfc-cc90-43da-841f-aa57d4d789c7,1.79m,82.4772kg,24,40.7,20.87%,Average,3.9727,NaN,2025-01-12,"Dr. sth., MD"
1,f7164dc2-6646-420e-b6b5-42fc07c11f6d,180.1135cm,93.8985,29,40.3,23.38,NaN,4.1549,NaN,2025-01-02,"Dr. sth., MD"
2,09c9de13-5d3a-4b75-a7ec-ce10bb32efdd,163.1483cm,79.2438,25,37.2,23.41,good,4.0825,NaN,2025-01-17,Mr. Many good sounding titles
3,be8969bf-6de2-49fd-b0e9-c014bdb970d6,1.82m,80.8913kg,29,42.5,21.28%,Good,3.5001,NaN,2025/01/05,unintelligible handwriting
4,476cd0f0-0154-4e70-be7c-661007e7fcc4,167.7038cm,109.2016,25,37.3,31.71,excellent,3.7619,NaN,2025-01-02,unintelligible handwriting


In [35]:
def parse_height(x):
    """Return height in metres. Handles '1.79m', '180.1135cm', bare numbers, NaN."""
    if pd.isna(x):
        return np.nan
    s = str(x).strip().lower()
    if "nan" in s:
        return np.nan
    m = re.match(r"([\d.]+)\s*(cm|m)?", s)
    if not m:
        return np.nan
    val = float(m.group(1))
    return val / 100 if m.group(2) == "cm" else val

def parse_weight(x):
    """Return weight in kg. 'nankg' and similar tokens become NaN."""
    if pd.isna(x):
        return np.nan
    s = str(x).strip().lower()
    if "nan" in s:
        return np.nan
    m = re.match(r"([\d.]+)", s)
    return float(m.group(1)) if m else np.nan

def parse_pct(x):
    if pd.isna(x):
        return np.nan
    s = str(x).strip().rstrip("%")
    try:
        return float(s)
    except ValueError:
        return np.nan

medical_clean["height"]              = medical_clean["height"].map(parse_height)
medical_clean["weight"]              = medical_clean["weight"].map(parse_weight)
medical_clean["age_in_years"]        = pd.to_numeric(medical_clean["age_in_years"], errors="coerce")
medical_clean["shoe_size"]           = pd.to_numeric(medical_clean["shoe_size"], errors="coerce")
medical_clean["body_fat_percentage"] = medical_clean["body_fat_percentage"].map(parse_pct)
medical_clean["sprint_time"]         = pd.to_numeric(medical_clean["sprint_time"], errors="coerce")
medical_clean["fitness_level"]       = (
    medical_clean["fitness_level"].astype("string").str.lower().str.strip()
)
medical_clean["return_date"]         = pd.to_datetime(medical_clean["return_date"], errors="coerce")

medical_clean.dtypes

medical_id                     object
height                        float64
weight                        float64
age_in_years                  float64
shoe_size                     float64
body_fat_percentage           float64
fitness_level          string[python]
sprint_time                   float64
medical_information            object
return_date            datetime64[ns]
physician_signature            object
dtype: object

### 3.7 `hockey_scouting_notes` — normalize categorical text

In [36]:
scouting_clean = scouting.copy()
for col in ["position", "dominant_hand", "experience_level"]:
    scouting_clean[col] = (
        scouting_clean[col].astype("string").str.lower().str.strip()
    )
scouting_clean[["position", "dominant_hand", "experience_level"]].nunique()

position            5
dominant_hand       2
experience_level    4
dtype: int64

### 3.8 `performance` — sanity check `pass_completion_rate`

Already numeric; just verify the rate column is consistent with `passes_completed / passes_attempted`.

In [37]:
perf_clean = perf.copy()
perf_clean = perf_clean.rename(columns={"penality_minutes": "penalty_minutes"})

mask = perf_clean["passes_attempted"] > 0
recomputed = (
    perf_clean.loc[mask, "passes_completed"]
    / perf_clean.loc[mask, "passes_attempted"] * 100
).round(2)
diff = (recomputed - perf_clean.loc[mask, "pass_completion_rate"]).abs()
print(f"rows where reported rate differs from recomputed by >1pp: {(diff > 1).sum()}")

rows where reported rate differs from recomputed by >1pp: 0


## 4. Persist clean tables

In [38]:
contracts_clean.to_csv(OUT / "contracts.csv",  index=False)
scouting_clean.to_csv (OUT / "scouting.csv",   index=False)
identity.to_csv       (OUT / "identity.csv",   index=False)
medical_clean.to_csv  (OUT / "medical.csv",    index=False)
moms_clean.to_csv     (OUT / "moms.csv",       index=False)
perf_clean.to_csv     (OUT / "performance.csv",index=False)

for f in sorted(OUT.iterdir()):
    print(f.name, "-", f.stat().st_size, "bytes")

contracts.csv - 480139 bytes
identity.csv - 835577 bytes
medical.csv - 1250113 bytes
moms.csv - 910331 bytes
performance.csv - 1675188 bytes
scouting.csv - 875876 bytes


## 5. Next steps

- Join `identity` (international_id ↔ medical_id) with `medical_clean` on `medical_id`.
- Join everything else on `international_id`.
- Match `moms_clean` to identities on `(first_name, last_name, age, birth_city)` — there is no shared id.
- Outlier handling on `performance` (e.g. `shooting_percentage`, `salary` log-distribution).
- Imputation strategy for the remaining nulls (slide 6–7 material).